In [119]:
import os, numpy as np, h5py, json, torch, torch.nn as nn, torchvision, re
import matplotlib.pyplot as plt, torchvision.transforms as transforms
from skimage.io import imread
from skimage.transform import resize
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from collections import Counter
from random import seed, choice, sample
from PIL import Image
from torch.nn.utils.rnn import pack_padded_sequence

In [120]:
def save_checkpoint(epoch, encoder, decoder, decoder_optimizer):
    state = {'epoch': epoch,
             'encoder': encoder,
             'decoder': decoder,
             'decoder_optimizer': decoder_optimizer}
    filename = "checkpoint" + str(epoch) + ".pth"
    torch.save(state, filename)
    print("Checkpoint saved to {}".format(filename))

class AverageMeter(object):
    def __init__(self):
        self.val = self.avg = self.sum = self.count = 0

    def update(self, val, n=1):
        self.val, self.sum, self.count  = val, self.sum+val*n, self.count+n
        self.avg = self.sum / self.count

def adjust_learning_rate(optimizer, shrink_factor):
    optimizer.param_groups[0]['lr'] = optimizer.param_groups[0]['lr'] * shrink_factor
    print("The new learning rate is {:.3f}".format(optimizer.param_groups[0]['lr']))

In [121]:
def accuracy(scores, targets, k):
    """
    scores: (batch_size, max(decoding_length), vocab_size)
    targets: (batch_size, max(decoding_length))
    """
    batch_size = targets.size(0)
    _, ind = scores.topk(k, 1)
    correct = ind.eq(targets.view(-1, 1).expand_as(ind))
    correct_total = correct.view(-1).float().sum()
    return correct_total.item() * (100.0 / batch_size)

In [122]:
from torchvision.models import resnet101, ResNet101_Weights

weights = ResNet101_Weights.DEFAULT
model = resnet101(weights=weights)
preprocess = weights.transforms()
print(preprocess)

ImageClassification(
    crop_size=[224]
    resize_size=[232]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [123]:
print(preprocess)

ImageClassification(
    crop_size=[224]
    resize_size=[232]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


# I. Preprocessing

## 1. Vocabulary for embedding

In [124]:
# --- Vocabulary ---
class Voc:
    def __init__(self):
        self.word2index = {"<pad>": 0, "<start>": 1, "<end>": 2, "<unk>":3}
        self.index2word = {0: "<pad>", 1: "<start>", 2: "<end>", 3: "<unk>"}
        self.num_words = 4  # Count SOS/EOS/UNK

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.num_words
            self.index2word[self.num_words] = word
            self.num_words += 1

# --- Caption preprocessing ---
def preprocess_caption(caption):
    caption = caption.lower()
    caption = re.sub(r'[^a-z ]', '', caption)
    caption = '<start> ' + caption + ' <end>'
    return caption

def seq2ind(voc, sentence):
    indices = []
    for word in sentence.split(' '):
        if word in voc.word2index:
            indices.append(voc.word2index[word])
        else:
            indices.append(voc.word2index["<unk>"])
    return indices

## 2. Dataset Class for train_loader

In [125]:
# --- Dataset class ---
class Flickr8kCSV(Dataset):
    def __init__(self, root, ann_file, voc, transform=None):
        self.root = root
        self.transform = transform
        self.voc = voc
        self.samples = []

        # Load captions + preprocess
        with open(ann_file, "r", encoding="utf-8") as f:
            next(f)  # bỏ header
            for line in f:
                img_name, caption = line.strip().split(",", 1)
                caption = preprocess_caption(caption)
                self.voc.addSentence(caption)  # update vocab
                caption_idx = seq2ind(self.voc, caption)
                caption_len = len(caption_idx)
                self.samples.append((img_name, caption_idx, caption_len))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_name, caption_idx, caption_len = self.samples[idx]
        img_path = os.path.join(self.root, img_name)
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, caption_idx, caption_len

## 3. Create dataset

In [126]:
# --- Transform ---
transform = transforms.Compose([
    transforms.Resize((224, 224)),            # Resize ảnh về 224x224
    transforms.ToTensor(),                     # PIL -> Tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# --- Khởi tạo vocab + dataset ---
voc = Voc()
dataset = Flickr8kCSV(root="./Flickr8k/Images", ann_file="./Flickr8k/captions.txt", voc=voc, transform=transform)

In [127]:
from torch.nn.utils.rnn import pad_sequence
def collate_fn(batch):
    images, captions, lengths = zip(*batch)  # unzip batch

    # Stack images
    images = torch.stack(images, 0)          # (batch_size, 3, 224, 224)

    # Chuyển captions sang tensor và pad
    captions = [torch.tensor(c) for c in captions]
    captions_padded = pad_sequence(captions, batch_first=True, padding_value=0)  # <pad>=0

    lengths = torch.tensor(lengths)

    return images, captions_padded, lengths

train_loader = DataLoader(dataset, batch_size=10, shuffle=True, pin_memory=True,collate_fn=collate_fn)

In [128]:
img, caption_idx, length = next(iter(train_loader))
print(length)
print(length.shape)
print(caption_idx)
print(caption_idx.shape)  # (batch_size, 3, 224, 224)

tensor([17, 15, 17, 12, 15, 12, 10, 20, 10, 19])
torch.Size([10])
tensor([[   1,    4,   78,   38,  171,  473,   32,    4,  742,  299,  389,    4,
         2474,  547, 1057,   18,    2,    0,    0,    0],
        [   1,   26,  588,    9,  272,  144,    6,   26, 1752,   38,  568,  780,
          169,   18,    2,    0,    0,    0,    0,    0],
        [   1,    4,    5,   69,  265,   18,  132,    4,  199,   13,  200,  978,
           34, 3311, 1343,   18,    2,    0,    0,    0],
        [   1,    4,  149,    6,    4, 1705,  177,  118,    4, 1940,   18,    2,
            0,    0,    0,    0,    0,    0,    0,    0],
        [   1,   26,   19,   38,    4, 1504,  135,    9,  158,   77,   41,   26,
          655,   18,    2,    0,    0,    0,    0,    0],
        [   1,   49,  285,   50,  121,  263,    4, 1143,  677,  284,   18,    2,
            0,    0,    0,    0,    0,    0,    0,    0],
        [   1,  258,   53,  126, 1334, 1481,   47,  359,   18,    2,    0,    0,
            0,    0

# II. Image Captioning Model

## 1. Encoder

In [129]:
from torchvision.models import resnet101, ResNet101_Weights
class Encoder(nn.Module):
    def __init__(self):
        super(Encoder, self).__init__()
        resnet = resnet101(weights=ResNet101_Weights.IMAGENET1K_V1)
        all_modules = list(resnet.children())
        modules = all_modules[:-2] 
        self.resnet = nn.Sequential(*modules)
        self.avgpool = nn.AvgPool2d((7, 7))
        self.fine_tune()
    
    def fine_tune(self, fine_tune=False):
        for p in self.resnet.parameters():
            p.requires_grad = fine_tune

    def forward(self, images):
        """
        images: (batch_size, 3, 224, 224)
        """
        batch_size = images.shape[0]
        encoded_image = self.resnet(images) 
        global_features = self.avgpool(encoded_image).view(batch_size, -1)
        return global_features  # (batch_size, 2048)


## 2. Decoder

In [130]:
class Decoder(nn.Module):
    def __init__(self, embed_dim, decoder_dim, vocab_size, encoder_dim = 2048):
        super(Decoder, self).__init__()
        self.embed_dim = embed_dim
        self.decoder_dim = decoder_dim
        self.vocab_size = vocab_size
        self.encoder_dim = encoder_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTMCell(embed_dim + encoder_dim, decoder_dim)
        self.fc = nn.Linear(decoder_dim, vocab_size)
        self.init_weights()
    
    def init_weights(self):
        self.embedding.weight.data.uniform_(-0.1, 0.1)
        self.fc.weight.data.uniform_(-0.1, 0.1)
        self.fc.bias.data.fill_(0)

    def init_hidden_states(self, batch_size):
        h = torch.zeros(batch_size, self.decoder_dim)
        c = torch.zeros(batch_size, self.decoder_dim)
        return h, c

    def forward(self, global_image, encoded_captions, caption_lengths):
        """
        global_image: (batch_size, encoder_dim)
        encoded_captions: (batch_size, max_caption_length)
        caption_lengths: (batch_size)
        """
        batch_size = global_image.size(0)
        encoder_dim = global_image.size(-1)
        caption_lengths, sort_ind = caption_lengths.sort(dim=0, descending=True)
        global_image = global_image[sort_ind] # (batch_size, encoder_dim)
        encoded_captions = encoded_captions[sort_ind] # (batch_size, max_caption_length)
        embeddings = self.embedding(encoded_captions)  # (batch_size, max_caption_length, embed_dim)
        h,c = self.init_hidden_states(batch_size)  # (batch_size, decoder_dim)
        decode_lengths = (caption_lengths - 1).tolist()  # bỏ <end>, kdl list []
        predictions = torch.zeros(batch_size, max(decode_lengths), self.vocab_size)

        for t in range(max(decode_lengths)):
            batch_size_t = sum([l > t for l in decode_lengths])
            lstm_input = torch.cat([embeddings[:batch_size_t, t, :], global_image[:batch_size_t]], dim = -1)
            h, c = self.lstm(lstm_input, (h[:batch_size_t], c[:batch_size_t]))  # (batch_size_t, decoder_dim)
            preds = self.fc(h)  # (batch_size_t, vocab_size)
            predictions[:batch_size_t, t, :] = preds
            # predictions: (batch_size, max(decode_lengths), vocab_size)
        return predictions, encoded_captions, decode_lengths, sort_ind

# III. Define Training and evaluation

## 1. Training function and hyperparameters

In [131]:
def train(train_loader, encoder, decoder, crieton, decoder_optimizer, epoch, print_freq = 10, device="cpu"):
    encoder.train()
    decoder.train()

    losses = AverageMeter()
    top3accs = AverageMeter()

    for i, (img, caption, length) in enumerate(train_loader):
        img, caption, length = img.to(device), caption.to(device), length.to(device)
        global_features = encoder(img)  # (batch_size, 2048)
        scores, caps_sorted, decode_lengths, sort_ind = decoder(global_features, caption, length)
        targets = caps_sorted[:, 1:]  # bỏ <start>
        scores = pack_padded_sequence(scores, torch.tensor(decode_lengths, dtype=torch.int64), batch_first=True)
        targets = pack_padded_sequence(targets, torch.tensor(decode_lengths, dtype=torch.int64), batch_first=True)
        
        loss = crieton(scores.data, targets.data)
        decoder_optimizer.zero_grad()
        loss.backward()
        decoder_optimizer.step()

        top3 = accuracy(scores.data, targets.data, 3)
        losses.update(loss.item(), sum(decode_lengths)) 
        top3accs.update(top3, sum(decode_lengths))

        if i % print_freq == 0:
            print("Epoch: [{0}][{1}/{2}]\t"
                  "Loss {loss.val:.4f} ({loss.avg:.4f})\t"
                  "Top-3 Accuracy {top3.val:.3f} ({top3.avg:.3f})".format(
                   epoch, i, len(train_loader), loss=losses,
                   top3=top3accs))

In [132]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embed_dim = 512
decoder_dim = 512
decoder_lr = 5e-4
epochs = 1
batch_size = 60
print_freq = 100
checkpoint = None
rev_word_map = {k: v for k, v in voc.index2word.items()}

In [133]:
encoder = Encoder()
decoder = Decoder(embed_dim=embed_dim, decoder_dim=decoder_dim, vocab_size=voc.num_words)
encoder = encoder.to(device)
decoder = decoder.to(device)
decoder_optimizer = torch.optim.Adam(decoder.parameters(), lr=decoder_lr)
crieton = nn.CrossEntropyLoss().to(device)

## 2. Evaluation function

In [134]:
import cv2
from torchvision import transforms
from PIL import Image

def greedy_decode(image_path):
    decoder.eval()
    encoder.eval()
    sampled = []

    img = cv2.imread(image_path)  # Read image using OpenCV
    if img is None:
        raise ValueError(f"Could not load image at {image_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB
    img = Image.fromarray(img)  # Convert to PIL Image for torchvision transforms
    img_tensor = transform(img)  # Apply transforms, shape: [3, 224, 224]
    image = img_tensor.unsqueeze(0)  # Add batch dimension, shape: [1, 3, 224, 224]

    global_features = encoder(image.to(device))  # (1, 2048)
    pred = torch.LongTensor([[voc.word2index['<start>']]]).to(device)  # (1, 1)
    h, c = decoder.init_hidden_states(batch_size=1)  # (1, decoder_dim)

    for time_step in range(20):  # max sampling length
        embeddings = decoder.embedding(pred).squeeze(1)  # (1, embed_dim)
        lstm_input = torch.cat([embeddings, global_features], dim=-1)  # (1, embed_dim + encoder_dim)
        h, c = decoder.lstm(lstm_input, (h, c))  # (1, decoder_dim)
        preds = decoder.fc(h)  # (1, vocab_size)
        _, pred = preds.max(1)  # (1)
        sampled.append(pred.item())
        if pred == voc.word2index['<end>']:
            break
        pred = pred.unsqueeze(1)  # (1, 1)
    
    generated_caption = ' '.join([rev_word_map[ind] for ind in sampled])
    return generated_caption

# IV. Training và evaluation

## 1. Training

In [ ]:
for epoch in range(epochs):
    if epoch % 3 == 0: adjust_learning_rate(decoder_optimizer, 0.8)
    train(train_loader, encoder, decoder, crieton, decoder_optimizer, epoch, print_freq, device)
    save_checkpoint(epoch, encoder, decoder, torch.optim.Adam(decoder.parameters(), lr=decoder_lr))

## 2. Evaluation

In [ ]:
greedy_decode("./Flickr8k/Images/1000268201_693b08cb0e.jpg")
# My device does not support CUDA so i dont train the model
# It take about 1 hour to train just 1 EPOCH on my laptop
# You can try to run on Google Colab with GPU